# The API

Sends photos to the HTTP API (`check_api.py`) and shows what comes back. `POST /v1/check` answers in one of two formats, chosen by the request's `Accept` header:

- **JSON** (the default, for programs): every check and every rule with its status, reasons and measurements.
- **HTML** (`Accept: text/html`, for people): the same report as one self-contained page, with the photo, the detected face and the planned 35×45 crop drawn in. Save it, open it in a browser or attach it to an issue.

Errors are always JSON with a stable `code`, whatever `Accept` says.

By default this notebook runs the API inside the notebook through FastAPI's `TestClient`: the same app and the same HTTP handling, without a server to start. To call a running server instead, start it with `uv run passport-validator-api` and set `PASSPORT_API_URL=http://127.0.0.1:9600` before starting Jupyter.

In [ ]:
import contextlib
import html
import mimetypes
import os
from pathlib import Path

import httpx
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent
IMAGES = ROOT / 'tests' / 'test-images'
REPORTS = ROOT / 'reports'  # gitignored
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

# OFIQ cloned next to this repo, unless OFIQ_CONFIG_DIR says otherwise; without OFIQ most rules are skipped
OFIQ_DATA = ROOT.parent / 'OFIQ-Project' / 'data'
if 'OFIQ_CONFIG_DIR' not in os.environ and OFIQ_DATA.is_dir():
    os.environ['OFIQ_CONFIG_DIR'] = str(OFIQ_DATA)



@contextlib.contextmanager
def quiet_native_logs():
    """Hide MediaPipe's C++ log lines, which go straight to the process's stderr."""
    saved = os.dup(2)
    with open(os.devnull, 'w') as devnull:
        os.dup2(devnull.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(saved, 2)
            os.close(saved)


API_URL = os.environ.get('PASSPORT_API_URL')
stack = contextlib.ExitStack()
if API_URL:
    client = stack.enter_context(httpx.Client(base_url=API_URL, timeout=120))
else:
    from fastapi.testclient import TestClient
    from edutap.image_api.check_api import app
    # Entering the client runs the app's startup, which loads the validator
    with quiet_native_logs():
        client = stack.enter_context(TestClient(app))
print('API:', API_URL or 'inside this notebook (TestClient)', '·', client.get('/health').json())


def check(photo, accept=None):
    """POST a photo to /v1/check; accept is the Accept header (None sends none, which means JSON)."""
    headers = {'Accept': accept} if accept else {}
    content_type = mimetypes.guess_type(photo.name)[0] or 'application/octet-stream'
    with open(photo, 'rb') as file, quiet_native_logs():
        return client.post('/v1/check', files={'file': (photo.name, file, content_type)}, headers=headers)


def show_page(page, height=900):
    """Show an HTML page in a frame, so its styles stay out of the notebook."""
    frame = f'<iframe srcdoc="{html.escape(page)}" allow="clipboard-write" style="width:100%;height:{height}px;border:1px solid #d0d7de"></iframe>'
    display({'text/html': frame}, raw=True)

## The rule catalogue

`GET /v1/rules` lists every rule of the Fotomustertafel, the checks that cover it, and how well (`full`, `partial` or `none`).

In [ ]:
catalogue = client.get('/v1/rules').json()
GROUPS = {group['id']: group['title'] for group in catalogue['groups']}
pd.DataFrame([
    {'group': GROUPS[rule['group']], 'rule': rule['title'], 'coverage': rule['coverage'], 'checks': ', '.join(rule['checks'])}
    for rule in catalogue['rules']
])

## Check a photo: JSON

Without an `Accept` header the answer is JSON. Set `PHOTO` to any photo to try your own.

In [ ]:
PHOTO = IMAGES / '9.jpg'

response = check(PHOTO)
report = response.json()
print(response.status_code, response.headers['content-type'])
print('Verdict:', 'valid' if report['is_valid'] else 'not valid', '· score', report['overall_score'])
print('Rules that could not be checked:', report['skipped_rules'] or 'none')
print('Fields:', ', '.join(report))

rules = pd.DataFrame([
    {'group': GROUPS[rule['group']], 'rule': rule['title'], 'status': rule['status'], 'reasons': '; '.join(rule['reasons'])}
    for rule in report['rules']
])
rules[rules['status'] != 'pass']

When a framing check (format, head size, face position) warns or fails, the report also carries `crop`: the 35×45 box the validator would cut inside the photo, and what the checks say about the cropped photo. The verdict above is still for the photo as sent.

In [ ]:
crop = report['crop']
print('Crop to 35x45:', crop['status'])
if crop['status'] == 'proposed':
    print(f"  box: x={crop['x']} y={crop['y']} {crop['width']}×{crop['height']} px")
    print('  the cropped photo is', 'valid' if crop['is_valid'] else 'still not valid')
    print('  fixes:', ', '.join(crop['fixed_rules']) or 'nothing')
    print('  still blocking:', ', '.join(crop['remaining_rules']) or 'nothing')
elif crop['status'] == 'not_possible':
    print(' ', crop['reason'])

Every check, with what it measured:

In [ ]:
pd.DataFrame([
    {'check': result['name'], 'status': result['status'], 'reasons': '; '.join(result['reasons']),
     'measurements': ', '.join(f'{key}={value:.4g}' if isinstance(value, float) else f'{key}={value}'
                               for key, value in result['measurements'].items())}
    for result in report['checks']
])

## The same photo as an HTML report

With `Accept: text/html` the same request returns the report as a page. It is saved to `reports/` and shown below. Its Copy JSON and Download JSON buttons hand out the JSON answer from above.

In [ ]:
response = check(PHOTO, accept='text/html')
REPORTS.mkdir(exist_ok=True)
path = REPORTS / f'{PHOTO.stem}.html'
path.write_text(response.text, encoding='utf-8')
print(response.status_code, response.headers['content-type'], '·', f'{len(response.content) / 1024:.0f} KB', '·', path)
show_page(response.text)

From a terminal, with the server running (`uv run passport-validator-api`):

```bash
# JSON
curl -F "file=@photo.jpg" http://127.0.0.1:9600/v1/check
# HTML report, saved to report.html
curl -F "file=@photo.jpg" -H "Accept: text/html" -o report.html http://127.0.0.1:9600/v1/check
```

A browser that posts a form sends `Accept: text/html` on its own, so it gets the page too.

## Errors carry a code

An error is JSON with a `code` to branch on and a `detail` for people, even when the request asked for HTML.

In [ ]:
for name, data, content_type in [('notes.txt', b'not a photo', 'text/plain'), ('empty.jpg', b'', 'image/jpeg')]:
    response = client.post('/v1/check', files={'file': (name, data, content_type)}, headers={'Accept': 'text/html'})
    print(response.status_code, response.headers['content-type'], response.json())

## A report for every test photo

Saves one HTML report per photo in `tests/test-images` to `reports/`. Large photos take several seconds each.

In [ ]:
saved = []
for photo in sorted(p for p in IMAGES.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}):
    response = check(photo, accept='text/html')
    path = REPORTS / f'{photo.stem}.html'
    path.write_text(response.text, encoding='utf-8')
    saved.append({'photo': photo.name, 'status': response.status_code, 'report': str(path.relative_to(ROOT))})
pd.DataFrame(saved)